# Validation of interpolation and ejection fraction

Run from the `notebooks/` folder after `pip install -e ".[notebooks]"`. Paths are relative to it.

What earlier evaluations measured, and what this notebook measures instead:

| Claim | Earlier evaluation | Here |
|---|---|---|
| Accurate interpolation between slices | dice of reconstructions *at the input slices* | dice of **held-out** slices, vs. copying the nearest slice |
| Accurate interpolation through the cardiac cycle | qualitative (one interpolated frame) | dice of **held-out frames** of a cine sequence |
| Ejection fraction estimation | EF of the implicit fit of the ground-truth masks vs. EF of those same masks | EF of the implicit fit of **automatic** segmentations vs. ground truth, with Bland–Altman agreement |

In [ ]:
import glob
import os

import numpy as np
import matplotlib.pyplot as plt

from implicit_heart import ImplicitFunction
from implicit_heart import metrics
from implicit_heart.validation import bland_altman, leave_one_out

SPACING = (10, 1.5625, 1.5625)  # ACDC voxel size (slice, y, x) in mm

## 1. Held-out slices (runs on the bundled data)

Each interior LV slice of the bundled end-diastolic volume is removed, the rest is fitted at the slices' physical positions, and the function is evaluated at the removed slice.

In [ ]:
M0 = np.load("../examples/data/M0.npy").astype(np.uint8)
slices = [z for z in range(len(M0)) if M0[z].any()]
coords = [z * SPACING[0] / SPACING[2] for z in slices]  # slice positions in pixel units

scores = leave_one_out(M0[slices], coords, grid_spacing=0)
interp, nearest = np.array(list(scores.values())).T
for i, (a, b) in scores.items():
    print(f"slice {slices[i]}: interpolated dice {a:.4f}   nearest slice {b:.4f}")
print(f"mean: interpolated {interp.mean():.4f}, nearest slice {nearest.mean():.4f}")

labels = [str(slices[i]) for i in scores]
x = np.arange(len(labels))
plt.bar(x - 0.2, interp, 0.4, label="implicit interpolation")
plt.bar(x + 0.2, nearest, 0.4, label="copy nearest slice")
plt.xticks(x, labels); plt.xlabel("held-out slice"); plt.ylabel("dice"); plt.ylim(0.8, 1); plt.legend(); plt.show()

## 2. Held-out frames of a cine sequence

Requires `../data/npy/lv.npy`: binary LV masks of every frame, shaped `(T, Z, H, W)`. Each held-out frame is interpolated from the remaining frames (one fit per held-out frame, so this takes a while).

In [ ]:
cine = np.load("../data/npy/lv.npy")
cine = (cine != 0).astype(np.uint8)
frames = np.arange(len(cine), dtype=float)              # time coordinate: frame index

scores = leave_one_out(cine, frames, held=range(1, len(cine) - 1, 2), grid_spacing=0, spacing=SPACING)
interp, nearest = np.array(list(scores.values())).T
print(f"mean dice: interpolated {interp.mean():.4f}, nearest frame {nearest.mean():.4f}")
plt.plot(list(scores), interp, "o-", label="implicit interpolation")
plt.plot(list(scores), nearest, "s--", label="copy nearest frame")
plt.xlabel("held-out frame"); plt.ylabel("dice"); plt.legend(); plt.show()

## 3. Ejection fraction from automatic segmentations

Requires the ACDC training set at `../data/acdc/training/patientXXX/` and the automatic segmentations of the same ED/ES frames from the segmentation pipeline, stored the same way: `../data/acdc_auto/patientXXX/patientXXX_frameYY_auto.nii.gz` with the LV cavity as label 3 (or set `AUTO_LV_LABEL`). The implicit function is fitted to the automatic masks only; the ground truth is used only for scoring.

In [ ]:
from implicit_heart.acdc import load_frames

AUTO_DIR, AUTO_PATTERN, AUTO_LV_LABEL = "../data/acdc_auto", "*_auto.nii.gz", 3

ef_auto, ef_gt, patients = [], [], []
for folder in sorted(glob.glob("../data/acdc/training/patient*")):
    patient = os.path.basename(folder)
    auto_folder = os.path.join(AUTO_DIR, patient)
    if not glob.glob(os.path.join(auto_folder, AUTO_PATTERN)):
        continue
    gt1, gt2, spacing = load_frames(folder)
    auto1, auto2, _ = load_frames(auto_folder, AUTO_PATTERN)
    auto = np.array([auto1 == AUTO_LV_LABEL, auto2 == AUTO_LV_LABEL]).astype(np.uint8)
    imp = ImplicitFunction(auto, tmax=1, grid_spacing=0, spacing=spacing)
    ef_auto.append(metrics.ejection_fraction_surface(imp))
    ef_gt.append(metrics.ejection_fraction(np.array([gt1 == 3, gt2 == 3])))
    patients.append(patient)
    print(f"{patient}: EF implicit(auto) {ef_auto[-1]:.1f}%   ground truth {ef_gt[-1]:.1f}%")

agreement = bland_altman(ef_auto, ef_gt)
print(f"n = {len(patients)}  bias {agreement['bias']:+.2f}  95% limits of agreement "
      f"[{agreement['lower']:+.2f}, {agreement['upper']:+.2f}]  r = {agreement['r']:.3f}")

In [ ]:
mean = (np.array(ef_auto) + np.array(ef_gt)) / 2
diff = np.array(ef_auto) - np.array(ef_gt)
plt.scatter(mean, diff, s=15)
for y, style in ((agreement["bias"], "-"), (agreement["lower"], "--"), (agreement["upper"], "--")):
    plt.axhline(y, color="gray", linestyle=style)
plt.xlabel("mean EF (%)"); plt.ylabel("implicit(auto) - ground truth (%)")
plt.title("Bland-Altman: ejection fraction"); plt.show()